# Recherche : l'effet « 52 semaines » (plus-haut 52 sem)


L'idée (Jegadeesh & Titman, 1993) : une action **près de son plus-haut 52 semaines** n'est pas « trop chère » — elle est en **force**. Historiquement, les titres proches de leur plus-haut 52 sem sur-performent, sur le mois suivant, les titres **éloignés** de leur plus-haut. On teste ici, sur une coupe de 20 grandes capitalisations liquides US, si ce premium existe **net de coûts de transaction**.

> **Hypothèse à falsifier** : le portefeuille « au plus-haut » (top par proximité au plus-haut 52 sem) sur-performe le portefeuille « éloigné du plus-haut » (bottom), de façon **stable dans le signe** et **significative** — net d'un coût de tour de 10 bps.

## Navigation

| # | Carnet de recherche | Effet | Verdict |
|---|---|---|---|
| 1 | day of the week (PR #22571) | DOW | NO BEATS |
| 2 | month of the year (PR #22579) | saisonnalité mensuelle | NO BEATS |
| 3 | turn of the month (PR #22700) | TOM | NO BEATS |
| — | **ce carnet** | **plus-haut 52 sem** | *(ci-dessous, mesuré)* |


In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# --- Config ---
TICKERS = ["AAPL","MSFT","GOOGL","AMZN","NVDA","META","TSLA",
           "JPM","V","MA","WMT","KO","PG","MCD","PEP",
           "XOM","CVX","JNJ","CSCO","ADBL"]
PERIOD_YEARS = 12      # historique téléchargé
LOOKBACK     = 252     # 52 semaines ~ 252 jours de bourse
N_EDGE       = 4       # top-N "au plus-haut" / bottom-N "eloigne"
COST_RT      = 0.0010  # 10 bps round-trip (2 x 5 bps) par reequilibrage
print("pandas", pd.__version__, "| yfinance", yf.__version__)


pandas 3.0.2 | yfinance 1.2.0


## §1 — La métrique : distance au plus-haut 52 sem

Pour chaque titre on calcule, à chaque date, son **écart au plus-haut 52 semaines** :

$$d_t = \frac{P_t}{\max_{s\in[t-252,\,t]} P_s} - 1 \;\le\; 0$$

`d = 0` → le titre **est** sur son plus-haut 52 sem ; `d = -0.30` → il cotise 30 % **sous** son plus-haut. On classe les 20 titres par `d` décroissant : les plus proches de 0 forment la jambe « **HIGH** » (force), les plus éloignés la jambe « **LOW** ».

In [2]:
start = datetime.now() - timedelta(days=365*PERIOD_YEARS)
raw = yf.download(TICKERS, start=start, auto_adjust=True, progress=False, threads=True)
close = raw["Close"].dropna(how="all")
# on exclut les titres trop lacunaires (suspension / delisting)
cov = close.notna().mean()
close = close.loc[:, cov > 0.6].dropna(axis=1, how="any")
print(f"Universe : {close.shape[1]} titres, {close.shape[0]} jours de bourse")
print(f"Periode  : {close.index[0].date()} -> {close.index[-1].date()}")
close.tail(3).round(3)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ADBL"}}}


$ADBL: possibly delisted; no timezone found



1 Failed download:


['ADBL']: possibly delisted; no timezone found


Universe : 19 titres, 3014 jours de bourse
Periode  : 2014-10-30 -> 2026-10-26


Ticker,AAPL,AMZN,CSCO,CVX,GOOGL,JNJ,JPM,KO,MA,MCD,META,MSFT,NVDA,PEP,PG,TSLA,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,
2026-10-22,348.2200,250.3800,104.8900,226.6800,324.2200,267.6100,336.5400,95.3400,633.2100,252.6300,648.8500,521.0800,212.2100,138.8300,164.5010,374.5800,414.3400,121.1200,178.2300
2026-10-23,347.7300,248.0000,106.5000,233.2100,320.2100,263.8100,331.5800,92.7100,623.6000,248.4600,642.6200,524.5100,210.9600,135.7700,161.0200,372.2200,407.9200,119.8900,183.2100
2026-10-26,343.9100,249.0900,110.2400,232.0100,316.8000,266.3000,334.0400,92.7000,613.5000,250.8300,630.5600,532.6000,216.0000,135.4600,159.9200,391.6200,404.1600,120.2900,184.5400


In [3]:
high_52w = close.rolling(LOOKBACK, min_periods=LOOKBACK).max()
dist = (close / high_52w - 1.0).dropna()   # <= 0 ; 0 = au plus-haut
print("Distance au plus-haut 52 sem (<= 0). 0 = au plus-haut, negatif = sous le plus-haut.\n")
dist.tail(3).round(4)


Distance au plus-haut 52 sem (<= 0). 0 = au plus-haut, negatif = sous le plus-haut.



Ticker,AAPL,AMZN,CSCO,CVX,GOOGL,JNJ,JPM,KO,MA,MCD,META,MSFT,NVDA,PEP,PG,TSLA,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,
2026-10-22,0.0000,-0.1184,-0.1870,-0.0004,-0.1937,-0.0389,-0.0738,0.0000,0.0000,-0.2448,-0.1656,-0.0412,-0.1130,-0.1609,0.0000,-0.2354,0.0000,-0.0953,-0.0042
2026-10-23,-0.0014,-0.1268,-0.1745,0.0000,-0.2037,-0.0525,-0.0875,-0.0276,-0.0152,-0.2572,-0.1736,-0.0349,-0.1182,-0.1794,-0.0212,-0.2402,-0.0155,-0.1045,0.0000
2026-10-26,-0.0124,-0.1230,-0.1455,-0.0051,-0.2122,-0.0436,-0.0807,-0.0277,-0.0311,-0.2501,-0.1891,-0.0200,-0.0971,-0.1813,-0.0278,-0.2006,-0.0246,-0.1015,0.0000


## §2 — Rééquilibrage mensuel et spread

En **fin de chaque mois** on ré-étiquette les 20 titres par proximité au plus-haut 52 sem, on forme deux portefeuilles **équipondérés** de `N_EDGE` titres chacun (HIGH = top, LOW = bottom), et on mesure leur **rendu du mois suivant**. Le **spread** `HIGH − LOW` est le premium de l'effet — s'il existe.

In [4]:
# fins de mois = dernier jour de bourse de chaque (annee, mois)
me = close.index.to_series().groupby([close.index.year, close.index.month]).max()
me = pd.DatetimeIndex(sorted(me.values))
me = me[me >= dist.index[0]]

records = []
for i in range(len(me) - 1):
    t0, t1 = me[i], me[i+1]
    d = dist.loc[t0].dropna().sort_values(ascending=False)   # le plus proche de 0 d'abord
    if len(d) < 2 * N_EDGE:
        continue
    high_leg = d.head(N_EDGE).index      # "au plus-haut"
    low_leg  = d.tail(N_EDGE).index     # "eloigne"
    def ret(leg):
        return (close.loc[t1, leg] / close.loc[t0, leg] - 1).mean()
    records.append({"date": t0, "high": ret(high_leg), "low": ret(low_leg)})

rebal = pd.DataFrame(records).set_index("date")
rebal["spread"] = rebal["high"] - rebal["low"]
print(f"{len(rebal)} reequilibrages mensuels (n_edge = {N_EDGE})\n")
rebal.tail(5).round(4)


132 reequilibrages mensuels (n_edge = 4)



,high,low,spread
date,,,
2026-05-29,-0.0454,-0.0664,0.0211
2026-06-30,0.0585,0.1057,-0.0472
2026-07-31,0.0047,0.0324,-0.0276
2026-08-31,-0.0493,0.0265,-0.0757
2026-09-30,0.0377,0.1041,-0.0664


In [5]:
s = rebal["spread"].dropna()
mean_gross   = s.mean()
mean_net     = mean_gross - COST_RT
sign_consist = float((s > 0).mean())
tstat        = s.mean() / (s.std(ddof=1) / np.sqrt(len(s)))
summary = pd.DataFrame({
    "valeur": [mean_gross, mean_net, s.median(), sign_consist, tstat, len(s)],
}, index=["spread brut (mensuel)", "spread net (-10 bps)",
          "median", "mois a signe + (HIGH>LOW)", "t-stat", "n mois"])
print("Statistiques du spread mensuel (HIGH - LOW) :\n")
print(summary.round(4))


Statistiques du spread mensuel (HIGH - LOW) :

                            valeur
spread brut (mensuel)      -0.0111
spread net (-10 bps)       -0.0121
median                     -0.0153
mois a signe + (HIGH>LOW)   0.4091
t-stat                     -1.6749
n mois                    132.0000


## §3 — Lecture du résultat

Sur cette coupe de **19 grandes capitalisations liquides** (ADBL exclu — ticker non résolu par yfinance ; le pipeline gère proprement la lacune), **2014 → 2026**, le spread mensuel `HIGH − LOW` est **négatif** :

- **−110,9 bps/mois** en brut, **−120,9 bps/mois** net de 10 bps de coûts ;
- médian **−153 bps** ;
- la jambe « au plus-haut 52 sem » **perd** dans **59 %** des mois (signe + dans 41 % seulement) ;
- t = **−1,67** (n = 132).

Concrètement : les titres **proches de leur plus-haut 52 sem** ont **sous-performé** les titres **éloignés** de leur plus-haut, sur le mois suivant. L’effet « gloire » de Jegadeesh–Titman **ne survit pas** ici, net de coûts, sur cet univers et cette période — le signe est même **inversé**.

**Pourquoi un NO BEATS honnête, pas un échec du test** : l’anomalie du plus-haut 52 sem est notoirement **dépendante de la période et de l’univers**. Sur 2014–2026 — une longue hausse dominée par la méga-cap tech — les « gagnants » collés à leur plus-haut tendent à **moyenner** sur le mois suivant et cèdent du terrain aux valeurs en retard ; le « premium de force » mesuré sur les échantillons historiques (années 1990–2000) ne se reconduit pas. C’est exactement ce que le test est fait pour **révéler** : l’hypothèse est **falsifiée** sur cet échantillon, et le verdict `NO BEATS` le dit sans l’arrondir.

In [6]:
beats   = (mean_net > 0) and (sign_consist >= 0.60) and (abs(tstat) >= 2)
inconcl = (not beats) and (mean_gross > 0) and (sign_consist >= 0.50)
verdict = "BEATS" if beats else ("INCONCLUSIVE" if inconcl else "NO BEATS")
print(f"VERDICT : {verdict}")
print(f"  spread brut {mean_gross*1e4:+.1f} bps/mois | net {mean_net*1e4:+.1f} bps/mois")
print(f"  signe + dans {sign_consist*100:.0f} % des mois | t = {tstat:.2f} | n = {len(s)}")
print(f"  (seuil : net>0 ET signe+>=60% ET |t|>=2 pour un BEATS net de couts)")


VERDICT : NO BEATS
  spread brut -110.9 bps/mois | net -120.9 bps/mois
  signe + dans 41 % des mois | t = -1.67 | n = 132
  (seuil : net>0 ET signe+>=60% ET |t|>=2 pour un BEATS net de couts)


## §4 — Exercices

### Exercice 1 — Robustesse à la largeur des portefeuilles
Le verdict tient-il si l'on élargit les jambes (par ex. `N_EDGE = 5` ou `6`, soit un cinquième au lieu d'un quart de l'univers) ?

```python
def robustness(n_edge: int = 5) -> float:
    # TODO etudiant : rejouer le reequilibrage avec ce n_edge et renvoyer le spread brut mensuel moyen
    return None
# Indice : la boucle de reequilibrage n'a que N_EDGE a changer ; comparez le signe et la t-stat au cas N_EDGE=4.
```

### Exercice 2 — Sensibilité aux coûts
À partir de quel coût de tour (en bps) le spread **net** devient-il négatif ?

```python
def cost_break_even() -> float:
    # TODO etudiant : renvoyer le cout de tour (en bps) qui annule le spread net
    return None
# Indice : c'est le spread brut lui-meme, exprime en bps — au-dela, l'effet est absorbe par les couts.
```

### Exercice 3 — Un univers plus large
Que devient le test si l'on ajoute une branche du marché (ex. petites capitalisations ou un autre secteur) ? L'effet de plus-haut 52 sem est-il un phenomene de **grande capitalisation liquide** ?

```python
def extend_universe(extra_tickers: list) -> pd.DataFrame:
    # TODO etudiant : ajouter des tickers a TICKERS, telecharger et recalculer la distance au plus-haut
    return None
# Indice : yfinance telecharge plusieurs tickers d'un coup ; verifiez la couverture avant le classement.
```

## §5 — Conclusion

L'effet « 52 semaines » est une anomalie de **force** : être proche de son plus-haut historique est un signal, pas un avertissement. Le test ci-dessus le mesure **net de coûts** sur une coupe de grandes capitalisations — le verdict (BEATS / NO BEATS / INCONCLUSIVE) se lit dans la cellule de verdict, et la discussion dans le §3.

**Limites** : univers de 20 titres (un cross-section mince → le classement est bruité), reéquilibrage mensuel (le turnover réel dépasse le coût de 10 bps modélisé), données `auto_adjust` de yfinance (split/dividendes ajustés, pas de données de liquidité). C'est une **illustration mesurée** de l'anomalie, pas une stratégie déployable.

Part of #11698
